# Poly x Kalshi Architecture Design

**Project:** Poly x Kalshi cross-market prediction arbitrage scanner  
**Current active product direction:** research-only cross-venue scanner for sports game/match prediction markets  
**Last updated:** 2026-06-27

This notebook documents the architecture of the current MVP: how market pairs are discovered, manually approved, monitored on GCP, scored for complementary-buy arbitrage, stored for backtesting, and surfaced in dashboard/reporting tools.

The system is intentionally **alert-only**. It does not place trades, manage private keys, run an exchange account, or attempt atomic execution. The goal of the current burn-in is to answer:

> Are there frequent, real, executable cross-market price gaps between equivalent Polymarket and Kalshi sports markets?


## Executive Summary

The architecture is a scheduled ETL and signal-scoring pipeline:

```text
manual approved mappings
        |
        v
Cloud Scheduler -> Cloud Run Job -> Polymarket + Kalshi REST orderbooks
                                      |
                                      v
                           normalized orderbook snapshots
                                      |
                                      v
                         complementary-buy signal scoring
                                      |
                                      v
                GCS raw/processed artifacts + Cloud Logging
                                      |
                                      v
                      Streamlit dashboard + viability reports
```

The current deployed cross-sports scanner uses **approved non-futures pairs only**. At the time this architecture note was written, the active mapping set contains:

| Market family | Approved pairs |
|---|---:|
| Tennis matches | 102 |
| World Cup games | 29 |
| MLB games | 24 |
| WNBA games | 6 |
| **Total** | **161** |

The cloud job polls both venues for every approved pair. Each tick creates two orderbook rows per pair, so `161` approved pairs produce `322` orderbook rows and `322` scored complementary-buy signal rows.


## Design Goals

1. **Research before execution**  
   The system should measure whether the strategy is viable before adding capital, keys, or execution complexity.

2. **Manual mapping as the safety gate**  
   Automated matching can suggest candidate pairs, but only `status=approved` rows are monitored for alerts.

3. **Cloud-scheduled, not always-on**  
   A one-shot Cloud Run Job triggered by Cloud Scheduler keeps cost and operational surface small.

4. **Appendable evidence trail**  
   Every snapshot writes run logs, normalized orderbooks, scored signals, and alerts so viability can be evaluated historically.

5. **No hidden trading assumptions**  
   Signal scoring includes buffers and depth checks, but it still represents informational opportunity detection, not guaranteed executable profit.


## Repository Map

| Path | Role |
|---|---|
| `src/prediction_market/fifa_arbitrage.py` | Core scanner: discovery, mapping suggestions, orderbook normalization, scoring, artifact writes, CLI entrypoints |
| `config/cross_sports_market_mappings.csv` | Manual approval gate for cross-sports monitoring |
| `infra/gcp/` | Terraform for Artifact Registry, Cloud Run Jobs, Cloud Scheduler, GCS, IAM |
| `cloudbuild.gcp-scanner.yaml` | Cloud Build recipe for scanner image |
| `docker/Dockerfile.gcp-scanner` | Container image used by Cloud Run Jobs |
| `apps/monitor_dashboard.py` | Streamlit dashboard over local or GCS scanner outputs |
| `src/prediction_market/dashboard_data.py` | Dashboard loaders, summaries, viability metrics, pair recommendations |
| `src/prediction_market/viability_report.py` | Non-interactive Markdown/CSV/JSON viability report generator |
| `data/cross_sports_arbitrage/` | Local raw/processed scanner outputs |
| `reports/` | Local pulled summaries and generated viability reports |


## Logical Architecture

```mermaid
flowchart LR
    A[Venue Market Discovery] --> B[Approval Candidates]
    B --> C[Suggested Mappings]
    C --> D[Manual Review]
    D --> E[Approved Mapping CSV]

    E --> F[Scheduled Snapshot Runner]
    F --> G[Polymarket CLOB Books]
    F --> H[Kalshi Orderbooks]
    G --> I[YES/NO Normalization]
    H --> I
    I --> J[Complementary-Buy Scoring]
    J --> K[Strategy Signals]
    J --> L[Arbitrage Alerts]
    F --> M[Scanner Run Log]

    K --> N[GCS / Local Processed Tables]
    L --> N
    M --> N
    N --> O[Dashboard]
    N --> P[Viability Report]
    N --> Q[Backtesting / Review]
```

The discovery path and the monitoring path are deliberately separate. Discovery may be noisy; monitoring is restricted to reviewed mappings.


## Cloud Deployment Architecture

```mermaid
flowchart TB
    subgraph Build
        A[Local repo] --> B[Cloud Build]
        B --> C[Artifact Registry image]
    end

    subgraph Runtime
        D[Cloud Scheduler: every 5 minutes] --> E[Cloud Run Job: sports snapshot]
        C --> E
        E --> F[Polymarket REST APIs]
        E --> G[Kalshi REST APIs]
        E --> H[GCS: cross_sports_arbitrage]
        E --> I[Cloud Logging]
    end

    subgraph Analysis
        H --> J[Streamlit dashboard]
        H --> K[Viability report]
        H --> L[Notebook review]
    end
```

Current cloud output path:

```text
gs://poly-x-kalshi-dev-poly-x-kalshi-scanner/cross_sports_arbitrage/
```

Current sports Cloud Run job:

```text
poly-x-kalshi-dev-sports-snapshot
```

Current sports scheduler:

```text
poly-x-kalshi-dev-sports-snapshot */5 * * * *
```


## Data Flow

### 1. Discovery

Discovery pulls active/open venue markets from:

- Polymarket Gamma events/markets, using sports tags and keyword filters.
- Kalshi Trade API events/markets, using sport-specific series tickers and keyword filters.

The scanner normalizes each venue market into `approval_candidates` with fields such as:

| Field | Meaning |
|---|---|
| `venue` | `polymarket` or `kalshi` |
| `market_type` | `match_winner`, `championship_winner`, `total`, `spread`, `other`, etc. |
| `event_title` | Normalized event name, for example `Seattle Mariners vs. Pittsburgh Pirates` |
| `event_date` | Event date inferred from venue payload/title |
| `event_match_key` | Date plus normalized teams/players, used for exact game matching |
| `outcome_label` | Team/player/Tie outcome represented by the market row |
| `settlement_summary` | Compact rules/title text for review |
| `liquidity_hint` | Venue-provided volume/liquidity fields when available |

### 2. Suggested mappings

`src/prediction_market/fifa_arbitrage.py` scores candidate pairs by:

- event key equality for game/match markets
- outcome equality after team/player alias normalization
- sport compatibility
- event date/start-time compatibility when available
- special tennis surname/team handling
- championship-scope guards for futures, although futures are excluded from the active run

Suggested rows remain `review_required`.

### 3. Manual approval

Only rows copied into `config/cross_sports_market_mappings.csv` with `status=approved` become eligible for monitoring.

Required review questions:

- Are the event and outcome identical?
- Is draw/Tie handling identical?
- Does the market include overtime, extra innings, extra time, or penalties?
- What happens if the event is postponed, canceled, or rescheduled?
- Are Polymarket token IDs and Kalshi ticker correct?


## Monitoring Snapshot

Each scheduled snapshot runs the same deterministic process:

```text
1. Load approved mappings
2. Fetch mapped Polymarket orderbooks
3. Fetch mapped Kalshi orderbooks
4. Normalize YES/NO bids, asks, and depths
5. Score complementary-buy directions
6. Write raw and processed artifacts
7. Print alert summaries to Cloud Logging
```

The active cloud job runs:

```bash
poly-x-kalshi-sports-snapshot \
  --no-discovery \
  --output-dir gs://poly-x-kalshi-dev-poly-x-kalshi-scanner/cross_sports_arbitrage \
  --mapping-path config/cross_sports_market_mappings.csv \
  --market-limit 1200 \
  --orderbook-depth 100 \
  --min-net-edge 0.02 \
  --slippage-buffer-per-leg 0.005 \
  --fee-buffer-total 0.01 \
  --min-depth-per-leg 10
```

`--no-discovery` is important: production monitoring should not accidentally start scoring unreviewed pairs.


## Price Normalization

The scanner converts both venues into comparable YES/NO bid/ask rows.

### Polymarket

Polymarket has CLOB outcome tokens. For an approved pair, the mapping row stores:

- `polymarket_yes_token_id`
- `polymarket_no_token_id`

The scanner pulls each token orderbook and records:

- `yes_bid`, `yes_ask`, `yes_bid_depth`, `yes_ask_depth`
- `no_bid`, `no_ask`, `no_bid_depth`, `no_ask_depth`

### Kalshi

Kalshi exposes YES and NO bid ladders. The scanner derives asks from opposite-side bids:

```text
yes_ask = 1 - best_no_bid
no_ask  = 1 - best_yes_bid
```

Depth for a derived ask comes from the opposite-side bid ladder. This keeps both venues in the same dollar-probability convention where `0.42` means 42 cents.


## Signal Model

For each approved mapping, the scanner scores two complementary-buy directions:

| Direction | Leg 1 | Leg 2 |
|---|---|---|
| `buy_polymarket_yes_buy_kalshi_no` | Polymarket YES ask | Kalshi NO ask |
| `buy_kalshi_yes_buy_polymarket_no` | Kalshi YES ask | Polymarket NO ask |

The conservative signal math is:

```text
gross_cost   = leg1_ask + leg2_ask
buffered_cost = gross_cost + 2 * slippage_buffer_per_leg + fee_buffer_total
net_edge     = 1 - buffered_cost
```

A row becomes an executable alert only when all of these are true:

```text
both asks are available
both legs have depth >= min_depth_per_leg
net_edge >= min_net_edge
```

Current default thresholds:

| Parameter | Value |
|---|---:|
| `min_net_edge` | `0.02` |
| `slippage_buffer_per_leg` | `0.005` |
| `fee_buffer_total` | `0.01` |
| `min_depth_per_leg` | `10` |

Interpretation: a pair must still have at least two cents of edge after one cent total slippage and one cent total fee buffer.


## Storage Layout

Local and GCS outputs share the same logical layout.

```text
cross_sports_arbitrage/
├── raw/
│   ├── polymarket/
│   └── kalshi/
├── processed/
│   ├── latest/
│   │   ├── manual_mappings_snapshot.{csv,parquet}
│   │   ├── orderbook_snapshots.{csv,parquet}
│   │   ├── arbitrage_alerts.{csv,parquet}
│   │   ├── strategy_signals.{csv,parquet}
│   │   └── scanner_runs.{csv,parquet}
│   ├── manual_mappings_snapshot/run_id=.../
│   ├── orderbook_snapshots/run_id=.../
│   ├── arbitrage_alerts/run_id=.../
│   ├── strategy_signals/run_id=.../
│   └── scanner_runs/run_id=.../
└── alerts/
    └── arbitrage_alerts.jsonl
```

`processed/latest/` is for dashboards and quick inspection. The partition-like `run_id=.../` directories are for historical analysis and viability review.


## Data Contracts

### `manual_mappings_snapshot`

One row per approved mapping used in a run.

| Field | Meaning |
|---|---|
| `mapping_id` | Stable pair identifier |
| `status` | Must be `approved` to run |
| `event_name` | Human-readable event |
| `proposition` | Outcome being mapped |
| `polymarket_market_id` | Polymarket condition/market ID |
| `polymarket_yes_token_id` | Token used for Polymarket YES leg |
| `polymarket_no_token_id` | Token used for Polymarket NO leg |
| `kalshi_ticker` | Kalshi market ticker |
| `draw_handling`, `extra_time_handling`, `penalties_handling` | Settlement-risk review notes |
| `settlement_notes` | Longer review context |

### `orderbook_snapshots`

Two rows per mapping: one Polymarket row and one Kalshi row.

| Field | Meaning |
|---|---|
| `run_id`, `retrieved_at`, `mapping_id`, `venue` | Snapshot identity |
| `yes_bid`, `yes_ask`, `no_bid`, `no_ask` | Normalized prices |
| `yes_bid_depth`, `yes_ask_depth`, `no_bid_depth`, `no_ask_depth` | Available size around best bid/ask |
| `raw_orderbook` | Compact JSON payload for audit/debug |
| `error` | Per-row retrieval/normalization error |

### `strategy_signals` and `arbitrage_alerts`

One row per mapping per direction.

| Field | Meaning |
|---|---|
| `direction` | Complementary-buy direction |
| `leg1_*`, `leg2_*` | Venue, outcome, ask, and depth for each leg |
| `gross_cost` | Sum of both asks before buffers |
| `buffered_cost` | Cost after slippage and fee buffers |
| `net_edge` | `1 - buffered_cost` |
| `is_alert` | Whether all filters passed |
| `exclusion_reason` | Why the row did not become an alert |

### `scanner_runs`

One row per snapshot run.

| Field | Meaning |
|---|---|
| `run_id` | Snapshot identity |
| `started_at`, `finished_at` | Runtime window |
| `status` | `succeeded` or failure state |
| `candidate_count` | Discovery rows, usually `0` in `--no-discovery` cloud mode |
| `approved_mapping_count` | Approved mappings loaded |
| `orderbook_count` | Orderbook rows written |
| `alert_count` | Number of executable alert rows |
| `error` | Run-level exception, if any |


## Observability

The current observability surface is intentionally simple:

| Layer | Tool |
|---|---|
| Scheduler health | `gcloud scheduler jobs describe poly-x-kalshi-dev-sports-snapshot` |
| Job health | `gcloud run jobs executions list` and `gcloud run jobs executions describe` |
| Runtime logs | Cloud Logging for `resource.type="cloud_run_job"` |
| Durable outputs | GCS `processed/latest/` and `processed/<table>/run_id=.../` |
| Product review | `poly-x-kalshi-dashboard` |
| Research review | `poly-x-kalshi-viability-report` and notebooks |

The job prints a JSON run summary and alert lines. Example fields:

```json
{
  "run_id": "sports-20260626T023655Z",
  "status": "succeeded",
  "approved_mapping_count": 161,
  "orderbook_count": 322,
  "alert_count": 0
}
```


## Current Burn-In Status

As of the latest cloud review on 2026-06-26:

- Cloud Scheduler is enabled for cross-sports snapshots every five minutes.
- The cloud job is polling `161` approved non-futures pairs.
- Recent runs are succeeding and writing `322` orderbook rows per tick.
- Early transient tennis alerts appeared shortly after deployment.
- Recent runs show `0` alerts, so there is no current material signal.

Important interpretation:

> The system has proven that it can collect and score cross-venue prices continuously. It has not yet proven that the strategy has persistent executable edge.


## Risk Register

| Risk | Why It Matters | Current Mitigation | Future Upgrade |
|---|---|---|---|
| Mapping mismatch | False arbitrage if markets settle differently | Manual approval fields for draw/overtime/cancellation | Review UI with rule diffs and pair lifecycle states |
| Stale/locked orderbooks | Large edge may be non-executable | Store raw payloads and depth; require min depth | Add quote freshness, halted/closed checks, venue status flags |
| Sequential polling latency | Prices may move before both legs are observed | Scheduled research-only snapshots | Concurrent fetcher and timestamp skew checks |
| No atomic execution | Complementary legs may not both fill | Alert-only MVP | Paper trading simulator before any execution layer |
| Fees and slippage uncertainty | Net edge may be overstated | Conservative fixed buffers | Venue-specific fee model and order-size simulation |
| Sports-specific settlement quirks | Draws, extra time, postponement rules differ | Required settlement notes in mapping CSV | Sport-specific validators and rule templates |
| Cloud image embeds config | Mapping changes require rebuild/redeploy | Explicit rebuild step | Load approved mappings from GCS/DB at runtime |
| GCS files as datastore | Fine for MVP, awkward for queries at scale | Appendable CSV/Parquet files | BigQuery external/native tables or DuckDB compaction |


## Why Cloud Scheduler + Cloud Run Jobs?

The MVP does not need an always-on worker yet.

| Option | Pros | Cons | MVP Decision |
|---|---|---|---|
| Local loop | Fast iteration, easy debugging | Fragile, not always-on | Used for development only |
| Cloud Scheduler + Cloud Run Jobs | Cheap, stateless, durable logs, simple retry model | Polling only, image rebuild for config | Current production burn-in |
| Always-on Cloud Run service | Can support API/dashboard/webhooks | More operational surface | Later, if alert delivery needs it |
| WebSockets | Lower latency | More state, reconnect logic, venue differences | Deferred until viability is proven |
| Pub/Sub + workers | Scales well | More moving parts | Later for multi-sport/high-frequency monitoring |


## Runbook

### Check scheduler state

```bash
gcloud scheduler jobs describe poly-x-kalshi-dev-sports-snapshot \
  --project poly-x-kalshi \
  --location us-central1 \
  --format='table(name,state,schedule,lastAttemptTime)'
```

### Pause cloud tracking

```bash
gcloud scheduler jobs pause poly-x-kalshi-dev-sports-snapshot \
  --project poly-x-kalshi \
  --location us-central1
```

### Resume cloud tracking

```bash
gcloud scheduler jobs resume poly-x-kalshi-dev-sports-snapshot \
  --project poly-x-kalshi \
  --location us-central1
```

### Run one manual cloud snapshot

```bash
gcloud run jobs execute poly-x-kalshi-dev-sports-snapshot \
  --project poly-x-kalshi \
  --region us-central1 \
  --wait
```

### Inspect latest GCS outputs

```bash
gcloud storage ls --long \
  gs://poly-x-kalshi-dev-poly-x-kalshi-scanner/cross_sports_arbitrage/processed/latest/
```

### Read recent job logs

```bash
gcloud logging read \
  'resource.type="cloud_run_job" AND resource.labels.job_name="poly-x-kalshi-dev-sports-snapshot"' \
  --project poly-x-kalshi \
  --freshness=2h \
  --limit=80 \
  --format='value(timestamp,textPayload)'
```

### Rebuild and push scanner image after config/code changes

```bash
IMAGE=us-central1-docker.pkg.dev/poly-x-kalshi/poly-x-kalshi-dev/fifa-scanner:latest

gcloud builds submit \
  --project poly-x-kalshi \
  --config cloudbuild.gcp-scanner.yaml \
  --substitutions "_IMAGE=$IMAGE" .
```


## Optional: Load Local Evidence Tables

The following cells are optional. They let the notebook display local pulled summaries such as `reports/cloud_signal_runs_snapshot.csv` and `reports/cloud_true_alerts_snapshot.csv` if those files exist.


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

reports_dir = PROJECT_ROOT / "reports"
runs_path = reports_dir / "cloud_signal_runs_snapshot.csv"
alerts_path = reports_dir / "cloud_true_alerts_snapshot.csv"

print("project root:", PROJECT_ROOT)
print("runs exists:", runs_path.exists())
print("alerts exists:", alerts_path.exists())


In [ ]:
if runs_path.exists():
    runs = pd.read_csv(runs_path)
    for column in ["started_at", "finished_at"]:
        if column in runs.columns:
            runs[column] = pd.to_datetime(runs[column], errors="coerce", utc=True)
    for column in ["candidate_count", "approved_mapping_count", "orderbook_count", "alert_count"]:
        if column in runs.columns:
            runs[column] = pd.to_numeric(runs[column], errors="coerce")

    display(runs.tail(10))
    print("run count:", len(runs))
    print("status counts:")
    print(runs["status"].value_counts(dropna=False).to_string())
    print("total alerts:", int(runs["alert_count"].fillna(0).sum()))
else:
    print("No local cloud_signal_runs_snapshot.csv found. Run the cloud pull command from the analysis workflow first.")


In [ ]:
if alerts_path.exists():
    alerts = pd.read_csv(alerts_path)
    if not alerts.empty:
        for column in ["detected_at", "detected_at_dt"]:
            if column in alerts.columns:
                alerts[column] = pd.to_datetime(alerts[column], errors="coerce", utc=True)
        for column in ["net_edge", "gross_cost", "min_depth", "leg1_ask", "leg2_ask"]:
            if column in alerts.columns:
                alerts[column] = pd.to_numeric(alerts[column], errors="coerce")

        summary = (
            alerts.groupby("mapping_id")
            .agg(
                alert_rows=("mapping_id", "size"),
                runs=("run_id", "nunique"),
                max_edge=("net_edge", "max"),
                median_edge=("net_edge", "median"),
                min_depth=("min_depth", "min"),
                median_depth=("min_depth", "median"),
            )
            .sort_values(["runs", "max_edge"], ascending=[False, False])
        )
        display(summary.head(25))
        display(alerts.sort_values("detected_at").tail(20))
    else:
        print("Alert file exists but has no true alert rows.")
else:
    print("No local cloud_true_alerts_snapshot.csv found.")


## Optional: Pull Cloud Summaries

This cell is guarded because it calls GCP. Set `RUN_CLOUD_ARCHITECTURE_CELLS=1` before running the notebook if you want it to copy the latest cloud files into `reports/`.


In [ ]:
import os
import subprocess
from pathlib import Path

RUN_CLOUD = os.getenv("RUN_CLOUD_ARCHITECTURE_CELLS") == "1"
GCS_ROOT = "gs://poly-x-kalshi-dev-poly-x-kalshi-scanner/cross_sports_arbitrage"

if not RUN_CLOUD:
    print("Skipping cloud calls. Set RUN_CLOUD_ARCHITECTURE_CELLS=1 to pull GCS summaries.")
else:
    reports_dir.mkdir(exist_ok=True)
    latest_dir = reports_dir / "cloud_latest"
    latest_dir.mkdir(exist_ok=True)
    for name in ["scanner_runs.csv", "arbitrage_alerts.csv", "strategy_signals.csv", "orderbook_snapshots.csv"]:
        source = f"{GCS_ROOT}/processed/latest/{name}"
        target = latest_dir / name
        print("copy", source, "->", target)
        subprocess.run(["gcloud", "storage", "cp", source, str(target)], check=True)
    print("done")


## Semantic Review Queue Architecture

The Review Queue now has two separate matching modes:

1. **Manual browse mode**  
   The dashboard shows separate active-event lists for Polymarket and Kalshi. The reviewer searches both sides, selects one row from each venue, fills settlement notes, and saves a decision.

2. **Suggested-pair mode**  
   The discovery job precomputes candidate pair suggestions to reduce the manual search space. Suggestions are not approvals. They are queue items for human review.

The intended architecture is:

```text
Polymarket active candidates        Kalshi active candidates
            |                              |
            v                              v
    canonical market rows          canonical market rows
            |                              |
            +--------------+---------------+
                           |
                           v
          deterministic hard gates and suggestion scoring
                           |
                           v
               suggested_mappings review table
                           |
                           v
              human approval / reject / needs_review
                           |
                           v
              approved mappings used by price monitor
```

The critical rule is that **manual approval remains the source of truth**. Neither local vector matching nor Gemini may directly add a pair to the monitoring set.


## Gemini Semantic Matching Layer

Gemini embeddings are a second-stage ranking tool for bulk review, not an execution signal.

Current command shape:

```bash
poly-x-kalshi-sports-snapshot \
  --discovery-only \
  --output-dir gs://poly-x-kalshi-dev-poly-x-kalshi-scanner/cross_sports_arbitrage \
  --mapping-path gs://poly-x-kalshi-dev-poly-x-kalshi-scanner/cross_sports_arbitrage/manual_review/approved_mappings/current.csv \
  --market-limit 1500 \
  --page-size 200 \
  --semantic-embedding-provider vertex-gemini \
  --semantic-embedding-dim 768 \
  --semantic-top-k 20 \
  --semantic-min-score 72
```

The semantic pipeline writes these artifacts:

| Table | Purpose |
|---|---|
| `approval_candidates` | One normalized candidate row per venue/outcome/market unit |
| `market_embeddings` | Cached canonical text vectors keyed by venue, market identifier, outcome, model, dimension, and text hash |
| `suggested_mappings` | Review queue suggestions from rules, local vectors, and Gemini semantic scoring |

Embedding cache behavior:

- Canonical text is hashed.
- If venue, identifier, outcome, model, dimension, and text hash match an existing row, the vector is reused.
- New Gemini vectors are flushed periodically to `processed/latest/market_embeddings.*` so long runs can be interrupted without losing all progress.
- Gemini is disabled by default in Terraform. It is enabled only when `sports_discovery_semantic_embedding_provider = "vertex-gemini"` or the local CLI flag is passed.


## Latest Semantic Discovery Result

Observed run:

```text
run_id: semantic-discovery-20260627T040059Z
candidate_count: 10,926
embedding_count: 10,926
suggested_mapping_count: 950
approved_mapping_count: 0
orderbook_count: 0
alert_count: 0
```

Suggestion breakdown from the saved `suggested_mappings` table:

| Suggestion method | Count | Interpretation |
|---|---:|---|
| `semantic` | 603 | Gemini embedding suggestions; useful for recall, currently too noisy |
| `rules` | 315 | Deterministic title/outcome/event-key matches; safest review bucket |
| `embedding` | 32 | Local hashed-vector suggestions; useful fallback, lower semantic quality |

Market type breakdown:

| Method + market type | Count | Review stance |
|---|---:|---|
| `semantic + total` | 600 | Too noisy right now; requires stricter metric and event gates |
| `rules + championship_winner` | 172 | Reviewable; must verify field/dead-heat/void rules |
| `rules + match_winner` | 121 | Reviewable; must verify same event and settlement definition |
| `embedding + match_winner` | 32 | Reviewable but lower confidence than rules |
| `rules + pole_position_winner` | 22 | Reviewable; must verify same race/session and official pole definition |
| `semantic + match_winner` | 3 | Currently suspicious because draw/team outcome semantics are not strict enough |

Because this was a discovery-only run, no orderbooks or price alerts were pulled. The result is a review queue, not a trading signal.


## Why The First Gemini Suggestions Were Noisy

The Gemini layer correctly made the candidate universe searchable, but it exposed a matching-design problem: semantic similarity alone is not enough for market equivalence.

Observed false-positive patterns:

- Tennis total-sets markets matched soccer/baseball total-goals or multi-outcome Kalshi titles.
- Draw/Tie Polymarket outcomes matched unrelated Kalshi team-winner markets.
- Event titles with similar sports language matched despite different participants.
- Bundled Kalshi titles containing several comma-separated legs looked semantically close to single Polymarket markets.
- `total` markets lost the unit distinction: sets, goals, points, runs, maps, rounds, and games are not interchangeable.

This is a useful architecture finding. The right role for Gemini is **ranking inside a narrow candidate set**, not replacing deterministic gates.


## Implemented Hard-Gate Design For Suggested Pairs

Before displaying rule, local-vector, or Gemini suggestions as review items, the pipeline now enforces these deterministic gates:

| Gate | Required behavior |
|---|---|
| Sport gate | Polymarket and Kalshi sport contexts must match exactly after alias normalization |
| Market type gate | `match_winner`, `championship_winner`, `total`, `spread`, `pole_position_winner`, etc. cannot cross-match |
| Date gate | If both sides expose an event date, dates must match or fall inside a venue-specific tolerance window |
| Participant gate | Match-winner markets require the same participant set, not just similar sports language |
| Outcome gate | Team/player outcomes must match the same participant; `Tie`/`Draw` can only match a draw/tie market |
| Totals unit gate | Totals must share the same metric: goals with goals, sets with sets, runs with runs, points with points |
| Threshold gate | Totals must share the same line, for example `Over 2.5` cannot match `Over 3.5` |
| Scope gate | Full game, first half, set, map, round, regular season, tournament, and qualification markets cannot cross-match |
| Bundle gate | Reject titles that look like multi-leg baskets or comma-separated parlays |
| Settlement gate | If settlement text contains mismatch clues such as regular-time vs advancement, mark `needs_review` or exclude |

Internal exclusion reasons used by the stricter gate:

| Status | Meaning |
|---|---|
| `review_required` | Candidate passes hard gates and is suitable for manual review |
| `needs_review` | Candidate has unresolved settlement ambiguity |
| `excluded_wrong_sport` | Sport contexts disagree |
| `excluded_wrong_event` | Dates or participants disagree |
| `excluded_wrong_market_type` | Market types disagree |
| `excluded_wrong_metric` | Totals/spreads have different metric units or lines |
| `excluded_bundle` | Candidate appears to be a basket/parlay/multi-leg market |

This design keeps recall high enough for review while protecting the queue from obviously non-equivalent pairs.


## Review Queue UX After Guardrails

The Review Queue should show three sections:

1. **High-confidence suggestions**  
   Rows that pass strict hard gates. Default sort should prioritize `rules`, then high-score `embedding`, then high-score `semantic`.

2. **Needs-review suggestions**  
   Rows that are plausible but have settlement ambiguity. These should be visually separated from high-confidence rows.

3. **Manual browse lists**  
   Separate active Polymarket and Kalshi tables with independent keyword search. This remains the fallback when suggestions miss a pair.

A suggested row should show:

| Field group | Examples |
|---|---|
| Identity | venue IDs, Polymarket slug, Kalshi ticker, URLs |
| Event | sport, league/category, event title, date, participant set |
| Market semantics | market type, outcome label, metric unit, threshold line, scope |
| Settlement review | draw/overtime/extra-time/penalty/cancellation notes |
| Scores | `match_score`, `embedding_score`, `gemini_embedding_score`, `semantic_combined_score`, `suggestion_method` |
| Exclusion/review reason | concise explanation of why the pair is accepted into review or excluded |

The approve button should continue writing only to the mapping store. The live scanner should continue reading only `status=approved` and `lifecycle_status=active` mappings.


## Dashboard Architecture

The dashboard is intentionally read-only. It consumes the same processed tables that the cloud job writes.

```text
GCS or local data directory
        |
        v
prediction_market.dashboard_data.load_dashboard_tables
        |
        v
summarize_dashboard / filter_signals / pair_history
        |
        v
apps.monitor_dashboard Streamlit UI
```

Dashboard tabs map to architecture concerns:

| Tab | Purpose |
|---|---|
| Overview | Latest run health, best current edges, alert rows |
| Viability | Historical alert rate, positive-edge rate, pair recommendations, sensitivity |
| Coverage | Discovery funnel and keyword coverage |
| Signals | All scored complementary-buy rows |
| Pairs | Approved mapping gate and pair-level history |
| Orderbooks | Normalized YES/NO book rows |
| Discovery | Candidate and suggestion review tables |
| Runs | Scanner run history |


## Next Architecture Upgrades

Recommended next steps, in order:

1. **Move mapping config out of the image**  
   Load `cross_sports_market_mappings.csv` from GCS so approvals do not require image rebuilds.

2. **Concurrent orderbook fetcher**  
   Current polling is sequential. Concurrency would reduce timestamp skew and make larger pair universes practical.

3. **Quote freshness and market-status filters**  
   Add explicit stale/closed/locked-market exclusion reasons before treating a large edge as material.

4. **Pair lifecycle states**  
   Add `monitor`, `watch`, `pause`, `expired`, and `needs_review` states to approved mappings.

5. **BigQuery or compacted Parquet layer**  
   GCS files are fine for burn-in, but historical querying will be easier with BigQuery external/native tables or a compacted daily Parquet layout.

6. **Alert delivery**  
   Add Slack/email/webhook only after the signal quality filters are stricter.

7. **Paper execution simulator**  
   Before real trading, simulate fill assumptions, leg risk, slippage, fees, and cancellation rules.

8. **Execution layer, if ever justified**  
   Only after repeated material signals survive paper execution review.


## Architecture Verdict

The current architecture is the right shape for the research stage:

- It is cheap to run.
- It has a durable evidence trail.
- It keeps human review in front of signal generation.
- It can be paused quickly.
- It avoids premature trading infrastructure.

The main architectural bottleneck is not cloud orchestration; it is **signal quality** and **market equivalence confidence**. The next build work should improve mapping lifecycle management, quote freshness checks, and historical viability analytics before adding any execution capability.
